# Logical compiler to rotated-surface PPM

This notebook compiles the integration programs and uses Stim's native detector-slice diagrams to show complete syndrome-extraction windows.


In [ ]:
import copy
from pathlib import Path
import sys

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'pyproject.toml').is_file() and (path / 'lightstim').is_dir()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import SVG, display

from examples.integrations.logical_compiler_rotated_surface_ppm import (
    compile_program,
    load_program,
)
from lightstim.qec_code.surface_code.rotated.ppm import UnsupportedPauliError

PROGRAM_DIR = ROOT / 'examples/integrations/logical_compiler_rotated_surface_ppm'


In [ ]:
def show_compilation(result):
    circuit = result.circuit
    print(f'{circuit.num_qubits} physical qubits, {circuit.num_ticks} ticks, '
          f'{circuit.num_detectors} detectors, {circuit.num_observables} observables')
    for index, plan in enumerate(result.experiment.plans):
        exact = plan.certificate is not None and plan.certificate.measures_exactly_the_product
        print(f'PPM {index}: {plan.kind}, schedule={plan.schedule}, exact_product={exact}')


def measurement_ticks(circuit):
    tick = 0
    result = []
    for instruction in circuit.flattened():
        if instruction.name == 'TICK':
            tick += 1
        elif instruction.name in {'M', 'MX', 'MY'} and (not result or result[-1] != tick):
            result.append(tick)
    return result


def show_detector_range(circuit, ticks):
    print(f'detector slices: ticks {ticks.start}-{ticks.stop - 1}')
    diagram = circuit.diagram('detslice-with-ops-svg', tick=ticks)
    display(SVG(str(diagram)))


## Two sequential weight-2 PPMs

Three collinear patches execute `M(Z_A Z_B)` followed by `M(Z_B Z_C)`.


In [ ]:
sequential_program = load_program(PROGRAM_DIR / 'sequential_ppm.json')
sequential_result = compile_program(sequential_program)
show_compilation(sequential_result)


Detector slices from initialization through the end of the first merged PPM syndrome-extraction round:


In [ ]:
sequential_measurements = measurement_ticks(sequential_result.circuit)
show_detector_range(
    sequential_result.circuit,
    range(0, sequential_measurements[1] + 1),
)


## One weight-3 PPM

A T-shaped corridor performs the true three-body measurement `M(Z_q1 Z_q2 Z_q3)`.


In [ ]:
weight3_program = load_program(PROGRAM_DIR / 'weight3_ppm.json')
weight3_result = compile_program(weight3_program)
show_compilation(weight3_result)


Detector slices from initialization through the end of the first merged PPM syndrome-extraction round:


In [ ]:
weight3_measurements = measurement_ticks(weight3_result.circuit)
show_detector_range(
    weight3_result.circuit,
    range(0, weight3_measurements[1] + 1),
)


## Larger 3x3 validation program

Nine patches execute three non-overlapping, non-nearest-neighbor PPMs with weights 2, 2, and 3. They cover pure `ZZ`, pure `XX`, and mixed `XZX`; q20 uses the explicit conjugate stabilizer frame required by the mixed join. `rounds=1` keeps each physical PPM window compact enough to inspect.


In [ ]:
checkerboard_program = load_program(PROGRAM_DIR / 'checkerboard_3x3_ppm.json')
checkerboard_result = compile_program(checkerboard_program, rounds=1)
show_compilation(checkerboard_result)


### Complete physical window for each PPM

Each detector-slice diagram spans a complete merge/split operation rather than one tick. Adjacent windows share the boundary tick at which one PPM finishes and the next begins.


In [ ]:
checkerboard_measurements = measurement_ticks(checkerboard_result.circuit)
checkerboard_ranges = [
    range(0, checkerboard_measurements[2] + 1),
    range(checkerboard_measurements[2], checkerboard_measurements[4] + 1),
    range(checkerboard_measurements[4], checkerboard_measurements[6] + 1),
]
for index, ticks in enumerate(checkerboard_ranges):
    print(f'PPM {index}:')
    show_detector_range(checkerboard_result.circuit, ticks)


### Pauli support boundary

The current routed rotated-surface-code lowering supports `X` and `Z`, including mixed products such as `XZX`. A `Y` term requires a twist-defect or equivalent Y-wall construction, so the interface rejects it explicitly instead of silently producing an invalid decomposition.


In [ ]:
y_program = copy.deepcopy(checkerboard_program)
y_program['operations'][3]['ppm'][0][1] = 'Y'
try:
    compile_program(y_program, rounds=1)
except UnsupportedPauliError as error:
    print(f'{type(error).__name__}: {error}')
